# 02. Gen-3 분석 결과 종합

**Saturation Score**: 0.585 → 0.628 → **0.691** (부분 포화 달성)

Gen-3 핵심 분석 결과를 데이터 기반으로 요약한다.
- 담화 분석: LEX/DSC/PRAG + TTR + 담화구조
- 페르소나 분석: 6유형 + 경계값 분석
- 네트워크 분석: 29노드 co-occurrence 네트워크
- 통합: F1~F6 핵심 발견

In [ ]:
import json
import os
from collections import Counter

DATA_DIR = '../data'
ANALYSIS_DIR = '../analysis'

def load_json(path):
    with open(path, 'r') as f:
        return json.load(f)

# Gen-2 균등 샘플 데이터
posts = load_json(f'{DATA_DIR}/raw/gen2_stratified_posts.json')

# 분석 결과
style = load_json(f'{ANALYSIS_DIR}/discourse/style_analysis.json')
ttr = load_json(f'{ANALYSIS_DIR}/discourse/gen2/ttr_analysis.json')
discourse_struct = load_json(f'{ANALYSIS_DIR}/discourse/gen2/discourse_structure.json')
lex = load_json(f'{ANALYSIS_DIR}/discourse/gen2/lex_analysis.json')
prag = load_json(f'{ANALYSIS_DIR}/discourse/gen2/prag_analysis.json')
network = load_json(f'{ANALYSIS_DIR}/network/gen2/summary.json')
centrality = load_json(f'{ANALYSIS_DIR}/network/gen2/centrality.json')
persona = load_json(f'{ANALYSIS_DIR}/profiles/gen2/persona_types.json')
dokgo = load_json(f'{ANALYSIS_DIR}/profiles/gen2/dokgo_instability_analysis.json')
kappa = load_json(f'{ANALYSIS_DIR}/evolution/cohens_kappa.json')

print(f'분석 대상: {len(posts)}개 게시글, {network["agent_count"]}명 에이전트')

## 1. 담화 분석 결과

In [ ]:
# 격식성 분포
hon = style['overall_honorific_distribution']
total = sum(hon.values())
print('=== 격식성 분포 (Gen-2 균등 샘플) ===')
for k, v in sorted(hon.items(), key=lambda x: -x[1]):
    bar = '█' * int(v / total * 50)
    print(f'  {k:12} {v:>4}개 ({v/total*100:5.1f}%) {bar}')

print(f'\n반말 비율: {hon.get("informal", 0)/total*100:.0f}% — 구조적 특성 (F1 지지)')

In [ ]:
# LEX 코드 분포 (Gen-3 재적용)
print('=== LEX 코드 분포 (Gen-2 균등 샘플 재적용) ===')
if 'overall_distribution' in lex:
    dist = lex['overall_distribution']
    total_hits = sum(dist.values())
    for code, hits in sorted(dist.items(), key=lambda x: -x[1]):
        pct = hits / total_hits * 100 if total_hits > 0 else 0
        bar = '█' * int(pct / 2)
        print(f'  {code:15} {hits:>5}건 ({pct:5.1f}%) {bar}')
    print(f'\nLEX-AI: {dist.get("LEX-AI", 0)}건 — Gen-3 핵심 이슈 (탐지 실패)')
else:
    print('LEX 분석 데이터 구조 확인 필요')
    print(f'키: {list(lex.keys())[:5]}')

In [ ]:
# TTR 분석 결과 (Gen-3 신규)
print('=== TTR 어휘 다양성 분석 ===')
if 'overall' in ttr:
    overall = ttr['overall']
    print(f'  전체 corpus TTR: {overall.get("corpus_ttr", "N/A")}')
    print(f'  전체 mean TTR:   {overall.get("mean_ttr", "N/A")}')
    print(f'  총 토큰:         {overall.get("total_tokens", "N/A")}')
    print(f'  고유 토큰:       {overall.get("unique_tokens", "N/A")}')

print('\n=== 에이전트별 TTR (상위 활동 에이전트) ===')
if 'by_agent' in ttr:
    agents_ttr = ttr['by_agent']
    # 30개 이상 게시글 에이전트 필터
    active = [(name, data) for name, data in agents_ttr.items() 
              if data.get('post_count', 0) >= 10]
    active.sort(key=lambda x: x[1].get('corpus_ttr', 0))
    
    print(f'{"에이전트":20} {"게시글":>6} {"corpus":>8} {"mean":>8}  설명')
    print('-' * 70)
    for name, data in active:
        ct = data.get('corpus_ttr', 0)
        mt = data.get('mean_ttr', 0)
        pc = data.get('post_count', 0)
        note = '⚠️ 극단적 반복' if ct < 0.1 else ''
        print(f'  {name:18} {pc:>4}개  {ct:>7.4f}  {mt:>7.4f}  {note}')

In [ ]:
# 담화 구조 분석 결과 (Gen-3 신규)
print('=== 담화 구조 유형 분포 ===')
if 'structure_distribution' in discourse_struct:
    dist = discourse_struct['structure_distribution']
    total = sum(dist.values())
    for struct_type, count in sorted(dist.items(), key=lambda x: -x[1]):
        pct = count / total * 100
        bar = '█' * int(pct / 2)
        print(f'  {struct_type:30} {count:>4}개 ({pct:5.1f}%) {bar}')
    print(f'\n  body-only 비율: {dist.get("body-only", 0)/total*100:.1f}% — F6 발견')
    print(f'  완전구조(intro-body-conclusion): {dist.get("intro-body-conclusion", 0)/total*100:.1f}%')
elif 'overall' in discourse_struct:
    print(f'전체 게시글: {discourse_struct["overall"].get("total_posts", "?")}개')
    if 'structure_types' in discourse_struct['overall']:
        for k, v in discourse_struct['overall']['structure_types'].items():
            print(f'  {k}: {v}')
else:
    print(f'키: {list(discourse_struct.keys())[:5]}')

## 2. 페르소나 분석 결과

In [ ]:
# 페르소나 유형 분포
print('=== 페르소나 유형 분포 (Gen-2) ===')
type_counts = Counter()
boundary_agents = []

if isinstance(persona, dict):
    for name, data in persona.items():
        if isinstance(data, dict):
            ptype = data.get('primary_type', 'unknown')
            score = data.get('primary_score', 0)
            type_counts[ptype] += 1
            if 0.40 <= score <= 0.55:
                boundary_agents.append((name, ptype, score))

total_agents = sum(type_counts.values())
for ptype, count in type_counts.most_common():
    pct = count / total_agents * 100 if total_agents > 0 else 0
    bar = '█' * int(pct / 2)
    print(f'  {ptype:15} {count:>3}명 ({pct:5.1f}%) {bar}')

print(f'\n=== 경계값 에이전트 (0.40~0.55) ===')
print(f'  {len(boundary_agents)}/{total_agents}명 ({len(boundary_agents)/total_agents*100:.0f}%) — 분류 불안정 구간')
for name, ptype, score in boundary_agents:
    print(f'  {name:18} {ptype:15} score={score:.2f}')

In [ ]:
# 독고종철 불안정 분석
print('=== 독고종철 페르소나 불안정 분석 ===')
if 'root_cause' in dokgo:
    rc = dokgo['root_cause']
    print(f'  원인: {rc.get("description", "N/A")}')
    print(f'  Gen-1: {rc.get("gen1_type", "?")} (score {rc.get("gen1_score", "?")})')
    print(f'  Gen-2: {rc.get("gen2_type", "?")} (score {rc.get("gen2_score", "?")})')
elif 'analysis' in dokgo:
    a = dokgo['analysis']
    print(f'  분석: {json.dumps(a, ensure_ascii=False, indent=2)[:500]}')
else:
    print(f'  키: {list(dokgo.keys())}')

print(f'\n결론: F3을 "코드스위칭 브리지" → "다마당 활동 허브"로 재정의')

## 3. 네트워크 분석 결과

In [ ]:
# 네트워크 요약
print('=== 네트워크 요약 (Gen-3: co-occurrence 포함) ===')
print(f'  노드: {network["network_nodes"]}개')
print(f'  엣지: {network["network_edges"]}개')
print(f'  밀도: {network["density"]:.4f}')
print(f'  고립 노드: {network["isolate_nodes"]}개')
print(f'  총 상호작용: {network["total_interactions"]}건')

print('\n=== PageRank Top 5 ===')
for name, score in network['top5_pagerank']:
    bar = '█' * int(score * 100)
    print(f'  {name:18} {score:.4f} {bar}')

print('\n=== Degree Top 5 ===')
for name, degree in network['top5_degree']:
    bar = '█' * int(degree / 20)
    print(f'  {name:18} {degree:>4} {bar}')

## 4. 측정 신뢰도

In [ ]:
# Cohen's Kappa
print('=== Cohen\'s Kappa (격식성 축) ===')
print(f'  Kappa: {kappa.get("kappa", "N/A")}')
print(f'  관찰 일치율: {kappa.get("observed_agreement", "N/A")}')
print(f'  기대 일치율: {kappa.get("expected_agreement", "N/A")}')
print(f'  불일치: {kappa.get("disagreements", "N/A")}개')
print(f'\n해석: {kappa.get("interpretation", "담화 격식성과 페르소나 유형은 독립 차원")}')

## 5. Saturation Score 진화

In [ ]:
# Saturation Score 세대별 비교
scores = {
    '코드 포화(30%)':    [0.55, 0.60, 0.73],
    '주제 안정성(25%)':  [0.50, 0.60, 0.67],
    '삼각검증(25%)':     [0.70, 0.72, 0.75],
    '반론 해소(20%)':    [0.60, 0.59, 0.58],
}
weights = [0.30, 0.25, 0.25, 0.20]

print('=== Saturation Score 세대별 비교 ===')
print(f'{"차원":20} {"Gen-1":>8} {"Gen-2":>8} {"Gen-3":>8}  변화')
print('-' * 60)
for dim, vals in scores.items():
    change = vals[2] - vals[1]
    arrow = '↑' if change > 0 else '↓' if change < 0 else '→'
    print(f'  {dim:18} {vals[0]:>7.3f}  {vals[1]:>7.3f}  {vals[2]:>7.3f}  {arrow}{abs(change):+.3f}')

totals = []
for gen_idx in range(3):
    total = sum(list(scores.values())[i][gen_idx] * weights[i] for i in range(4))
    totals.append(total)

print('-' * 60)
print(f'  {"가중 합산":18} {totals[0]:>7.3f}  {totals[1]:>7.3f}  {totals[2]:>7.3f}  ↑{totals[2]-totals[1]:+.3f}')
print(f'\n  판정: 미포화 → 미포화 → ★ 부분 포화 달성 ★')

## 6. 핵심 발견 요약

| ID | 발견 | 삼각검증 | 세대 |
|----|------|---------|------|
| F1 | 에이전트 간 담화·페르소나·네트워크 패턴 구별 가능 | **3/3** | Gen-1~3 안정 |
| F2 | 발신형·수신형 분화 (RobertBot 4지표 수렴) | **3/3** | Gen-2~3 강화 |
| F3 | 독고종철 다마당 활동 허브 (코드스위칭 아님) | 2.5/3 | Gen-3 재정의 |
| F4 | AI스러움-참여도 비선형 관계 | 2.5/3 | Gen-1~3 유지 |
| F5 | 5클러스터 다층 담화 스타일 유형론 | 2/3 | Gen-3 재현 |
| F6 | 담화 구조 불완전성 (body-only 48.2%) | 2/3 | Gen-3 신규 |

### 확정 가능한 7개 주장
1. PER-CURIOUS(44.8%) 지배적 페르소나
2. 반말 60% 우세는 구조적 특성
3. PRAG-QUOTE(46.66%)가 주요 화용 전략
4. RobertBot은 4개 독립 지표에서 식별 가능
5. 게시글 48.2%가 도입부 없이 전개부만
6. 명시적 동의/반대 표현은 3.3%에 불과
7. LLM-담화 인과관계는 현재 확정 불가

### 미해소 긴장 4개
1. LEX-AI 역설 (AI 커뮤니티인데 AI 어휘 0건)
2. 커뮤니티 vs 게시판 (PRAG-AGREE+DISAGREE 3.3%)
3. 분류 신뢰도 (경계값 에이전트 28%)
4. 비교 준거 부재

In [ ]:
print('='*60)
print('  봇마당 연구 Gen-3 분석 종합')
print('='*60)
print(f'  Saturation Score: 0.691 (부분 포화)')
print(f'  분석 에이전트: 29명')
print(f'  분석 게시글: 330개 (균등 샘플)')
print(f'  핵심 발견: 6개 (F1-F6)')
print(f'  확정 주장: 7개')
print(f'  미해소 긴장: 4개')
print(f'  Contrarian 반론: 15개 (3세대)')
print(f'  분석 스크립트: 10개')
print('='*60)
print('  조건부 학술 보고서 작성 가능 판정')
print('='*60)